# 08 Population Density Map

## Purpose

This notebook creates a population density layer for Harris County using Census tract boundaries and ACS demographic data. The density layer is then combined with selected METRO route geometry to visualize whether major high-ridership routes overlap with denser parts of the Houston region.

## Inputs

- `data/raw/tiger_shapefiles/tl_2025_48_tract.zip`
- `data/processed/acs_harris_tracts.csv` or `data/acs_harris_tracts.csv`
- `data/processed/key_route_geometry.csv`

## Outputs

- `data/processed/harris_tracts_density.geojson`
- Population density map
- Population density map with selected METRO routes

## Why This Matters

The project's central question is whether Houston transit investment aligns with ridership demand and population density. This notebook builds the geographic demographic layer needed to compare transit corridors with where people live.

## 1. Import Libraries

Pandas is used for tabular ACS data. GeoPandas is used for Census tract geometries. Matplotlib is used for static map visualizations.

In [ ]:
import os

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

## 2. Load Census Tract Boundaries

The TIGER/Line tract file contains polygon boundaries for Census tracts in Texas. These geometries provide the map shapes needed to create a population density visualization.

In [ ]:
tracts = gpd.read_file("../data/raw/tiger_shapefiles/tl_2025_48_tract.zip")

print("All Texas tracts:", tracts.shape)
tracts.head()

## 3. Filter to Harris County

The full TIGER file contains tracts for all of Texas. This analysis focuses on Harris County, where most of METRO's core service area is located.

Harris County's county FIPS code is `201`.

In [ ]:
harris_tracts = tracts[
    tracts["COUNTYFP"] == "201"
].copy()

print("Harris County tracts:", harris_tracts.shape)
harris_tracts.head()

## 4. Load ACS Demographic Data

The ACS file contains Census tract-level demographic variables previously pulled from the Census API.

Key variables used here:

- `B01003_001E`: Total population
- `B19013_001E`: Median household income

In [ ]:
processed_acs_path = "../data/processed/acs_harris_tracts.csv"
raw_acs_path = "../data/acs_harris_tracts.csv"

if os.path.exists(processed_acs_path):
    acs = pd.read_csv(processed_acs_path)
else:
    acs = pd.read_csv(raw_acs_path)

print("ACS rows/columns:", acs.shape)
acs.head()

## 5. Create Matching GEOID Values

Census geography data is joined using `GEOID`, a unique identifier made from state, county, and tract codes.

This step ensures the ACS table and tract shapefile use the same GEOID format before merging.

In [ ]:
acs["GEOID"] = (
    acs["state"].astype(str).str.zfill(2)
    + acs["county"].astype(str).str.zfill(3)
    + acs["tract"].astype(str).str.zfill(6)
)

harris_tracts["GEOID"] = harris_tracts["GEOID"].astype(str)

acs[["NAME", "GEOID", "B01003_001E", "B19013_001E"]].head()

## 6. Merge Demographics with Tract Geometry

This merge attaches population and income variables to each Census tract polygon. The result is a geospatial dataset that can be mapped and analyzed.

In [ ]:
tracts_acs = harris_tracts.merge(
    acs,
    on="GEOID",
    how="left"
)

print("Merged rows/columns:", tracts_acs.shape)
print("Missing population values:", tracts_acs["B01003_001E"].isna().sum())
tracts_acs.head()

## 7. Clean Numeric Variables

ACS values are imported as text, so population and income are converted into numeric fields before calculations.

In [ ]:
tracts_acs["population"] = pd.to_numeric(
    tracts_acs["B01003_001E"],
    errors="coerce"
)

tracts_acs["income"] = pd.to_numeric(
    tracts_acs["B19013_001E"],
    errors="coerce"
)

tracts_acs[
    ["population", "income"]
].describe()

## 8. Calculate Population Density

The TIGER file includes `ALAND`, which represents land area in square meters.

Population density is calculated as:

`population / square kilometers`

This makes it easier to compare tracts of different physical sizes.

In [ ]:
tracts_acs["sq_km"] = tracts_acs["ALAND"] / 1_000_000

tracts_acs["pop_density"] = (
    tracts_acs["population"] / tracts_acs["sq_km"]
)

tracts_acs[
    ["GEOID", "population", "sq_km", "pop_density"]
].head()

## 9. Save Processed Density Dataset

The merged tract-density dataset is saved as GeoJSON so it can be reused in later notebooks and eventually in an interactive web map.

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

tracts_acs.to_file(
    "../data/processed/harris_tracts_density.geojson",
    driver="GeoJSON"
)

print("Saved: ../data/processed/harris_tracts_density.geojson")

## 10. Map Harris County Population Density

This map shows where population is concentrated across Harris County. Darker tracts represent higher population density.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

tracts_acs.plot(
    column="pop_density",
    cmap="OrRd",
    legend=True,
    ax=ax,
    linewidth=0
)

ax.set_title("Houston / Harris County Population Density")
ax.set_axis_off()

plt.tight_layout()
plt.show()

## 11. Load Selected METRO Route Geometry

The route geometry file was created in the previous notebook. It contains coordinate points for selected high-ridership METRO routes.

In [ ]:
route_geometry = pd.read_csv(
    "../data/processed/key_route_geometry.csv"
)

print("Route geometry rows/columns:", route_geometry.shape)
route_geometry.head()

## 12. Overlay Major Routes on Population Density

This map combines the population density layer with selected METRO route alignments.

The goal is to visually inspect whether major ridership corridors pass through or near dense Census tracts.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

tracts_acs.plot(
    column="pop_density",
    cmap="OrRd",
    legend=True,
    ax=ax,
    linewidth=0
)

for route_name in route_geometry["route_long_name"].unique():
    route = route_geometry[
        route_geometry["route_long_name"] == route_name
    ].sort_values("shape_pt_sequence")

    ax.plot(
        route["shape_pt_lon"],
        route["shape_pt_lat"],
        linewidth=2,
        label=route_name
    )

ax.set_title("Population Density with Major METRO Routes")
ax.set_axis_off()
ax.legend()

plt.tight_layout()
plt.show()

## Summary

This notebook created a Harris County population density layer and overlaid selected METRO route geometries.

The results show how major high-ridership corridors relate to population density patterns. This geospatial foundation supports later investment analysis by connecting route performance with the demographic geography of Houston.